# 04 · The transformer block and a tiny GPT

After this notebook you can build a GPT-style decoder from its parts (token and positional embeddings, the pre-norm block, the LM head), derive its parameter count and check it against the real thing, explain sinusoidal positions and RoPE, and *train* a character-level GPT on nursery rhymes on your CPU - then watch it go from random characters to rhyme-shaped text.

**Time:** ~35 min · **Runs:** offline on CPU in about a minute (the training cell is ~15-40 s on a laptop; longer on a busy machine) · **Needs:** [03 · Attention from scratch](03_attention_from_scratch.ipynb), [PyTorch essentials](../04_numpy_pandas_pytorch/03_pytorch_essentials.ipynb), [training loops](../06_deep_learning/02_training_loop_optimizers_lr_schedules.ipynb)

## Why this matters in interviews

- "Walk me through one transformer block" and "how many parameters does this model have?" are standard; the answer is a short formula you can derive live.
- Knowing *where* the parameters live (the MLP holds ~2/3 of them) explains LoRA target choices, quantisation priorities and MoE.
- Positional encoding and RoPE come up in every long-context question ("how would you extend 4k to 32k?").
- Having trained one yourself makes "next-token prediction" concrete: the loss starts at ln(vocab) and every improvement is visible in the samples.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tr01` | What is a transformer? |
| `tr02` | Walk me through one transformer block, step by step. |
| `tr37` | How does a token id become a vector before self-attention runs? |
| `tr07` | How does the model know word order? Explain positional encoding and RoPE. |
| `tr08` | What is the residual stream, and why do residual connections matter conceptually? |
| `tr09` | What does LayerNorm do, and how is RMSNorm different? |
| `tr04` | What is the feed-forward layer for, and why does it matter that it holds most of the parameters? |
| `tr23` | What does the unembedding / LM head do, and what is weight tying? |
| `tn02` | Explain next-token prediction as a training objective. Why is it enough? |

In [ ]:
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
rng = np.random.default_rng(0)
torch.set_num_threads(1)          # tiny matmuls: one thread is fastest and immune to a busy CPU
print("torch", torch.__version__)

## 1. From token id to vector: the embedding table

A token id is just a row number. The embedding layer is a learned matrix of shape `(vocab_size, d_model)`, and "embedding a token" means *reading row `id`*. Mathematically that equals multiplying a one-hot vector by the matrix - but a lookup is far cheaper.

In [ ]:
vocab_size, d_model = 10, 4
emb = nn.Embedding(vocab_size, d_model)
ids = torch.tensor([3, 7, 3])
looked_up = emb(ids)
one_hot = F.one_hot(ids, vocab_size).float()
assert torch.allclose(looked_up, one_hot @ emb.weight), "lookup == one-hot times the matrix"
assert torch.equal(looked_up[0], looked_up[2]), "same id -> same vector, whatever the context"
print("ids", ids.tolist(), "-> vectors of shape", tuple(looked_up.shape))

The same id always gives the same vector - the embedding of "bank" is identical in "river bank" and "bank loan". Context only enters through attention in the blocks. Attention itself has no idea of order (it is permutation-equivariant, notebook 03), so the model also needs **position information**.

## 2. Positions: sinusoidal encodings and RoPE

**Sinusoidal (original Transformer, 2017):** add a fixed vector to each position, made of sines and cosines at geometrically spaced frequencies: `PE[pos, 2i] = sin(pos / 10000^(2i/d))`, `PE[pos, 2i+1] = cos(...)`. Low dimensions oscillate fast, high dimensions slowly - like the hands of a clock.

In [ ]:
def sinusoidal_pe(n_pos, d):
    pos = np.arange(n_pos)[:, None]
    freq = 1.0 / 10000 ** (np.arange(0, d, 2) / d)             # one frequency per (sin, cos) pair
    pe = np.zeros((n_pos, d))
    pe[:, 0::2], pe[:, 1::2] = np.sin(pos * freq), np.cos(pos * freq)
    return pe, freq

pe, freq = sinusoidal_pe(64, 64)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
im = axes[0].imshow(pe, cmap="RdBu", aspect="auto", vmin=-1, vmax=1)
axes[0].set_xlabel("embedding dimension"); axes[0].set_ylabel("position")
axes[0].set_title("Sinusoidal positional encoding"); fig.colorbar(im, ax=axes[0])
for dim in [0, 10, 30]:
    axes[1].plot(pe[:, dim], label=f"dim {dim}")
axes[1].set_xlabel("position"); axes[1].set_ylabel("value"); axes[1].legend()
axes[1].set_title("Each dimension is a wave with its own frequency")
plt.tight_layout(); plt.show()

k = 4                                                           # a fixed offset
rot = np.zeros((64, 64))                                        # block-diagonal rotation by k * freq
for i, w in enumerate(freq):
    c, s = np.cos(k * w), np.sin(k * w)
    rot[2*i:2*i+2, 2*i:2*i+2] = [[c, s], [-s, c]]
assert np.allclose(pe[10 + k], rot @ pe[10]) and np.allclose(pe[40 + k], rot @ pe[40])
assert np.isclose(pe[5] @ pe[9], pe[20] @ pe[24]), "similarity depends only on the offset"
print("PE[pos + k] = R_k · PE[pos] for every pos: moving by k positions is a fixed rotation")

The asserts show the elegant property: shifting by k positions is the *same rotation* whatever the starting position, so the dot product between two position vectors depends only on their distance. GPT-2 simply *learns* a position table instead (what our tiny GPT does below). Modern models use **RoPE**, which applies that rotation idea directly inside attention.

**RoPE (rotary position embedding):** instead of adding a vector to the input, rotate each query and key - pair of dimensions by pair of dimensions - by an angle proportional to its position. Then q_m · k_n depends only on the *relative* offset m - n. Values are not rotated.

In [ ]:
def rope(x, pos, base=10000.0):
    """Rotate consecutive pairs (x0,x1), (x2,x3), ... of x by angle pos * theta_i."""
    d = x.shape[-1]
    theta = base ** (-np.arange(0, d, 2) / d)
    c, s = np.cos(pos * theta), np.sin(pos * theta)
    out = np.empty_like(x)
    out[0::2] = x[0::2] * c - x[1::2] * s
    out[1::2] = x[0::2] * s + x[1::2] * c
    return out

q, kv = rng.normal(size=64), rng.normal(size=64)
assert np.isclose(rope(q, 5) @ rope(kv, 3), rope(q, 12) @ rope(kv, 10)), "same offset (2) -> same score"
assert np.isclose(np.linalg.norm(rope(q, 7)), np.linalg.norm(q)), "a rotation keeps the length"
scores = np.array([[rope(q, m) @ rope(kv, n) for n in range(32)] for m in range(32)])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
v2 = np.array([1.0, 0.0])
for p in range(8):
    a = p * 0.45
    r = np.array([v2[0] * np.cos(a) - v2[1] * np.sin(a), v2[0] * np.sin(a) + v2[1] * np.cos(a)])
    axes[0].annotate("", xy=r, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=plt.cm.viridis(p / 7), lw=2))
    axes[0].text(*(1.12 * r), f"pos {p}", fontsize=8, ha="center")
axes[0].set_xlim(-1.3, 1.3); axes[0].set_ylim(-0.3, 1.3); axes[0].set_aspect("equal"); axes[0].axis("off")
axes[0].set_title("RoPE: the same pair of dims, rotated more at later positions")
im = axes[1].imshow(scores, cmap="RdBu")
axes[1].set_xlabel("key position n"); axes[1].set_ylabel("query position m")
axes[1].set_title("q·k after RoPE: constant along diagonals (depends on m - n)")
fig.colorbar(im, ax=axes[1]); plt.tight_layout(); plt.show()

| Scheme | Where it acts | Relative? | Used by |
|---|---|---|---|
| Learned absolute | added to the input embedding | no; hard limit at the trained length | GPT-2, BERT |
| Sinusoidal | added to the input embedding | partly (rotation property) | original Transformer |
| **RoPE** | rotates q and k in every attention layer | yes | Llama, Mistral, Qwen, Gemma, GPT-NeoX |
| ALiBi | adds a distance penalty to attention scores | yes | BLOOM, MPT |

RoPE is why context extension works by *rescaling the rotation frequencies* (position interpolation, NTK-aware scaling, YaRN) plus a little fine-tuning at the longer length. (Implementation note: Hugging Face's Llama rotates dimension i with i + d/2 - "rotate_half" - rather than neighbours; it is the same idea with the dimensions permuted.)

## 3. The pre-norm transformer block

```
x ──►(+)────────────────────►(+)──► out        the residual stream: x flows straight through,
  │   ▲                    │   ▲                and each sub-layer ADDS its contribution
  └► LN ► causal self-attn ┘   │
                           └► LN ► MLP (d -> 4d -> d, GELU)
```

Two sub-layers, each wrapped as `x = x + sublayer(norm(x))`:

1. **Attention - "communicate":** each token gathers information from earlier tokens.
2. **MLP - "compute":** each token processes what it gathered, independently (a 2-layer network, 4x wider in the middle). This is where most parameters, and much of the stored knowledge, live.

*Pre*-norm (normalise before each sub-layer, GPT-2 onwards) keeps an untouched identity path through the whole network, which makes deep stacks train stably; the 2017 original used post-norm (`norm(x + sublayer(x))`) and needed careful warm-up.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d, n_heads, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.qkv = nn.Linear(d, 3 * d)            # W_q, W_k, W_v fused into one matmul
        self.proj = nn.Linear(d, d)               # W_o
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=-1)
        q, k, v = (t.view(B, T, self.n_heads, d // self.n_heads).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)   # == our NumPy code (notebook 03)
        return self.drop(self.proj(y.transpose(1, 2).reshape(B, T, d)))

class Block(nn.Module):
    def __init__(self, d, n_heads, dropout=0.0):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(d), CausalSelfAttention(d, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))            # communicate
        x = x + self.mlp(self.ln2(x))             # compute
        return x

blk = Block(d=64, n_heads=4)
x = torch.randn(2, 10, 64)
print("block: input", tuple(x.shape), "-> output", tuple(blk(x).shape), "(same shape, so blocks stack)")

**LayerNorm vs RMSNorm.** LayerNorm subtracts the mean and divides by the standard deviation of each token's vector, then applies a learned scale and shift. RMSNorm (Llama, Mistral, Qwen) skips the mean and the shift: `x / sqrt(mean(x²) + eps) * g` - cheaper, and works as well.

In [ ]:
class MyRMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps, self.weight = eps, nn.Parameter(torch.ones(d))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

xn = torch.randn(3, 64) * 5 + 2
ln_out = nn.LayerNorm(64)(xn)
assert torch.allclose(MyRMSNorm(64)(xn), nn.RMSNorm(64, eps=1e-6)(xn), atol=1e-6), "matches torch.nn.RMSNorm"
print(f"LayerNorm output: mean {ln_out.mean(-1)[0]:+.3f}, std {ln_out.std(-1, unbiased=False)[0]:.3f} per token")
print(f"LayerNorm params: {sum(p.numel() for p in nn.LayerNorm(64).parameters())} (scale + shift) | "
      f"RMSNorm params: {sum(p.numel() for p in nn.RMSNorm(64).parameters())} (scale only)")

| | GPT-2 (2019) | Llama 3 (2024) |
|---|---|---|
| Norm | LayerNorm, pre-norm | RMSNorm, pre-norm |
| Positions | learned absolute table | RoPE inside attention |
| Attention | multi-head (MHA) | grouped-query (GQA) |
| MLP | Linear -> GELU -> Linear (4d) | SwiGLU: 3 matrices (gate, up, down), ~3.5d |
| Biases | yes | none |
| LM head | tied to the token embedding | untied in 8B/70B (tied in the small 3.2 models) |

## 4. The whole model: token + position embeddings -> N blocks -> norm -> LM head

The **LM head** ("unembedding") maps each final vector back to one score (logit) per vocabulary entry. **Weight tying** reuses the token-embedding matrix for it: the same vector that represents a token going in is used to score it coming out.

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, context, d=64, n_layers=2, n_heads=4, dropout=0.0):
        super().__init__()
        self.context = context
        self.tok_emb = nn.Embedding(vocab_size, d)
        self.pos_emb = nn.Embedding(context, d)          # learned absolute positions (GPT-2 style)
        self.blocks = nn.ModuleList([Block(d, n_heads, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight           # weight tying
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):                                # GPT-2 style: small normal init
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                 # (B, T, vocab): a next-token score at EVERY position
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

### The parameter-count formula

Per block: attention has W_q, W_k, W_v, W_o (4d² weights + 4d biases); the MLP has d x 4d and 4d x d (8d² + 5d); two LayerNorms (4d). So **12d² + 13d per block**, and in total

`params = V·d (tokens) + T·d (positions) + L·(12d² + 13d) + 2d (final norm)`   (+ V·d if the head is not tied)

In [ ]:
def gpt_params(V, T, d, L, tied=True):
    return V * d + T * d + L * (12 * d * d + 13 * d) + 2 * d + (0 if tied else V * d)

def count(model):
    return sum(p.numel() for p in model.parameters())    # shared (tied) tensors are counted once

m_test = TinyGPT(vocab_size=57, context=64, d=64, n_layers=2, n_heads=4)
assert count(m_test) == gpt_params(57, 64, 64, 2), (count(m_test), gpt_params(57, 64, 64, 2))
assert m_test.head.weight is m_test.tok_emb.weight, "tied: one tensor, two roles"
print(f"tiny GPT: formula {gpt_params(57, 64, 64, 2):,} == model {count(m_test):,}")

gpt2_small = gpt_params(V=50257, T=1024, d=768, L=12)
assert gpt2_small == 124_439_808, "the published parameter count of GPT-2 small (124M)"
print(f"GPT-2 small: formula gives {gpt2_small:,} (the published 124M) | untied it would be "
      f"{gpt_params(50257, 1024, 768, 12, tied=False):,}")

In [ ]:
def breakdown(V, T, d, L):
    return {"token embedding": V * d, "position embedding": T * d, "attention": L * (4 * d * d + 4 * d),
            "MLP": L * (8 * d * d + 5 * d), "norms": L * 4 * d + 2 * d}

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, (name, cfg) in zip(axes, [("GPT-2 small (124M)", (50257, 1024, 768, 12)), ("our tiny GPT (108k)", (57, 64, 64, 2))]):
    parts = breakdown(*cfg)
    total = sum(parts.values())
    ax.barh(list(parts), [100 * v / total for v in parts.values()], color="tab:blue")
    ax.set_xlabel("% of parameters"); ax.set_title(name); ax.invert_yaxis()
    for i, v in enumerate(parts.values()):
        ax.text(100 * v / total + 0.5, i, f"{100 * v / total:.1f}%", va="center", fontsize=8)
    ax.set_xlim(0, 75)
plt.tight_layout(); plt.show()

b = breakdown(50257, 1024, 768, 12)
mlp_share = b["MLP"] / (b["attention"] + b["MLP"] + b["norms"])
print(f"GPT-2 small: the MLP holds {mlp_share:.0%} of the non-embedding parameters; "
      f"tying saves {50257 * 768 / 1e6:.1f}M ({50257 * 768 / gpt2_small:.0%} of the model)")
assert 0.6 < mlp_share < 0.7

Two lessons interviewers look for: **the MLPs hold about two thirds of the non-embedding parameters** (that is why LoRA often targets MLP matrices too, and why mixture-of-experts replaces the MLP), and **in tiny models the embedding tables dominate** - weight tying matters most there (31% of GPT-2 small).

## 5. The data: 4.7 KB of nursery rhymes, one character per token

Traditional (public-domain) rhymes. A character-level vocabulary is tiny (~57 symbols), which keeps the model small; real models use the subword tokenizers of notebook 01.

In [ ]:
TEXT = """Twinkle, twinkle, little star,
How I wonder what you are!
Up above the world so high,
Like a diamond in the sky.
Twinkle, twinkle, little star,
How I wonder what you are!

Mary had a little lamb,
Its fleece was white as snow;
And everywhere that Mary went,
The lamb was sure to go.

Humpty Dumpty sat on a wall,
Humpty Dumpty had a great fall.
All the king's horses and all the king's men
Couldn't put Humpty together again.

Jack and Jill went up the hill
To fetch a pail of water.
Jack fell down and broke his crown,
And Jill came tumbling after.

Baa, baa, black sheep,
Have you any wool?
Yes sir, yes sir,
Three bags full;
One for the master,
And one for the dame,
And one for the little boy
Who lives down the lane.

Hickory, dickory, dock,
The mouse ran up the clock.
The clock struck one,
The mouse ran down,
Hickory, dickory, dock.

Little Miss Muffet
Sat on a tuffet,
Eating her curds and whey;
Along came a spider,
Who sat down beside her
And frightened Miss Muffet away.

Hey diddle diddle,
The cat and the fiddle,
The cow jumped over the moon;
The little dog laughed
To see such sport,
And the dish ran away with the spoon.

Little Bo-Peep has lost her sheep,
And doesn't know where to find them;
Leave them alone, and they'll come home,
Wagging their tails behind them.

Row, row, row your boat,
Gently down the stream.
Merrily, merrily, merrily, merrily,
Life is but a dream.

Old King Cole was a merry old soul,
And a merry old soul was he;
He called for his pipe, and he called for his bowl,
And he called for his fiddlers three.

Sing a song of sixpence,
A pocket full of rye.
Four and twenty blackbirds,
Baked in a pie.
When the pie was opened,
The birds began to sing;
Wasn't that a dainty dish,
To set before the king?

Three blind mice, three blind mice,
See how they run, see how they run.
They all ran after the farmer's wife,
Who cut off their tails with a carving knife.

Hot cross buns! Hot cross buns!
One a penny, two a penny,
Hot cross buns!

Rain, rain, go away,
Come again another day.

Pat-a-cake, pat-a-cake, baker's man,
Bake me a cake as fast as you can;
Pat it and prick it, and mark it with B,
And put it in the oven for baby and me.

Jack be nimble,
Jack be quick,
Jack jump over
The candlestick.

Little Jack Horner
Sat in the corner,
Eating a Christmas pie;
He put in his thumb,
And pulled out a plum,
And said, "What a good boy am I!"

Georgie Porgie, pudding and pie,
Kissed the girls and made them cry.

Ring-a-ring o' roses,
A pocket full of posies,
A-tishoo! A-tishoo!
We all fall down.

Diddle, diddle, dumpling, my son John,
Went to bed with his trousers on;
One shoe off, and one shoe on,
Diddle, diddle, dumpling, my son John.

The grand old Duke of York,
He had ten thousand men;
He marched them up to the top of the hill,
And he marched them down again.

Mary, Mary, quite contrary,
How does your garden grow?
With silver bells and cockle shells,
And pretty maids all in a row.

Little Boy Blue, come blow your horn,
The sheep's in the meadow, the cow's in the corn.
Where is the boy who looks after the sheep?
He's under a haystack, fast asleep.

Old Mother Hubbard
Went to the cupboard,
To give the poor dog a bone;
When she came there,
The cupboard was bare,
And so the poor dog had none.

There was an old woman who lived in a shoe,
She had so many children she didn't know what to do.
She gave them some broth without any bread,
Then kissed them all soundly and put them to bed.

Pease porridge hot,
Pease porridge cold,
Pease porridge in the pot,
Nine days old.

Polly put the kettle on,
Polly put the kettle on,
Polly put the kettle on,
We'll all have tea.

One, two, buckle my shoe;
Three, four, knock at the door;
Five, six, pick up sticks;
Seven, eight, lay them straight;
Nine, ten, a big fat hen.

Wee Willie Winkie runs through the town,
Upstairs and downstairs in his nightgown,
Rapping at the window, crying through the lock,
Are the children all in bed? It's past eight o'clock.

Peter Piper picked a peck of pickled peppers;
A peck of pickled peppers Peter Piper picked.

Doctor Foster went to Gloucester
In a shower of rain;
He stepped in a puddle,
Right up to his middle,
And never went there again.

Ladybird, ladybird, fly away home,
Your house is on fire and your children are gone.

Simple Simon met a pieman,
Going to the fair;
Said Simple Simon to the pieman,
Let me taste your ware.

Tom, Tom, the piper's son,
Stole a pig and away he run;
The pig was eat, and Tom was beat,
And Tom went howling down the street.

The itsy bitsy spider climbed up the water spout.
Down came the rain and washed the spider out.
Out came the sun and dried up all the rain,
And the itsy bitsy spider climbed up the spout again.
"""
print(f"{len(TEXT):,} characters, {len(TEXT.split())} words")

In [ ]:
chars = sorted(set(TEXT))
V = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(chars[i] for i in ids)
assert decode(encode("Humpty Dumpty")) == "Humpty Dumpty"

data = torch.tensor(encode(TEXT))
n_train = int(0.9 * len(data))
train_data, val_data = data[:n_train], data[n_train:]     # the last rhymes are never trained on
CONTEXT, BATCH = 64, 16
print(f"vocab size V = {V}: {''.join(chars)!r}")
print(f"train {len(train_data):,} chars | val {len(val_data):,} chars")

gen_batch = torch.Generator().manual_seed(0)
def get_batch(split, g=gen_batch):
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - CONTEXT - 1, (BATCH,), generator=g)
    x = torch.stack([d[i:i + CONTEXT] for i in ix])
    y = torch.stack([d[i + 1:i + CONTEXT + 1] for i in ix])   # the same window shifted by one
    return x, y

xb, yb = get_batch("train")
print("input :", repr(decode(xb[0, :24].tolist())))
print("target:", repr(decode(yb[0, :24].tolist())), " <- every position predicts the NEXT character")

One window of 64 characters gives **64 training examples at once**: position t predicts character t+1 from characters 1..t. The causal mask makes that possible in a single forward pass - this is why next-token prediction on raw text is such an efficient objective: the text is its own label.

## 6. Before training

With small random weights the model's logits are almost equal, so it predicts a nearly uniform distribution and the cross-entropy loss is ln(V). Its samples are random characters.

In [ ]:
torch.manual_seed(0)
model = TinyGPT(V, CONTEXT, d=64, n_layers=2, n_heads=4, dropout=0.0)
print(f"parameters: {count(model):,} (formula: {gpt_params(V, CONTEXT, 64, 2):,})")

gen_eval = torch.Generator().manual_seed(123)
EVAL = {s: [get_batch(s, gen_eval) for _ in range(4)] for s in ("train", "val")}   # fixed eval batches

@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {s: float(np.mean([model(x, y)[1].item() for x, y in EVAL[s]])) for s in EVAL}
    model.train()
    return out

@torch.no_grad()
def generate(model, prompt, n_new=160, temperature=0.7, seed=0):
    g = torch.Generator().manual_seed(seed)
    model.eval()
    idx = torch.tensor([encode(prompt)])
    for _ in range(n_new):
        logits, _ = model(idx[:, -model.context:])        # crop: learned positions only go up to CONTEXT
        probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1, generator=g)], dim=1)
    model.train()
    return decode(idx[0].tolist())

init_loss = estimate_loss()
print(f"initial loss: train {init_loss['train']:.3f}, val {init_loss['val']:.3f}  vs  ln(V) = {math.log(V):.3f}")
assert abs(init_loss["train"] - math.log(V)) < 0.2, "an untrained model is ~uniform over the vocabulary"
sample_before = generate(model, "Twinkle", n_new=120)
print("\nsample BEFORE training:\n" + sample_before)

## 7. Train it

AdamW with a one-cycle schedule (short warm-up, then decay). Each step: sample 16 random windows, forward, cross-entropy on all 16 x 64 positions, backward, update.

In [ ]:
STEPS, LR = 500, 8e-3
torch.manual_seed(0)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=0.1)
history = []
t0 = time.perf_counter()
for step in range(1, STEPS + 1):
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()
    if step % 50 == 0 or step == 1:
        history.append((step, *estimate_loss().values()))
    if step % 100 == 0:
        print(f"step {step:4d} | train {history[-1][1]:.3f} | val {history[-1][2]:.3f} | {time.perf_counter() - t0:5.1f}s")
train_time = time.perf_counter() - t0
print(f"trained {STEPS} steps in {train_time:.1f}s ({1000 * train_time / STEPS:.0f} ms/step)")

In [ ]:
hist = pd.DataFrame(history, columns=["step", "train", "val"])
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(hist["step"], hist["train"], label="train loss")
ax.plot(hist["step"], hist["val"], label="validation loss (unseen rhymes)")
ax.axhline(math.log(V), color="gray", ls=":", label=f"ln(V) = {math.log(V):.2f} (uniform guess)")
ax.set_xlabel("step"); ax.set_ylabel("cross-entropy (nats per character)")
ax.set_title("Tiny GPT on nursery rhymes: loss vs step"); ax.legend(); ax.grid(alpha=0.3)
plt.show()
best_val = hist.loc[hist["val"].idxmin()]
print(f"final train {hist['train'].iloc[-1]:.3f} | best val {best_val['val']:.3f} at step {int(best_val['step'])} | "
      f"final val {hist['val'].iloc[-1]:.3f}")
assert hist["train"].iloc[-1] < 0.6 * math.log(V), "training must cut the loss substantially"

Read the two curves separately. **Training loss** keeps falling: the model is learning (and increasingly memorising) the rhymes. **Validation loss** on rhymes it never saw falls at first - it learns English spelling, spaces, capital letters after newlines - then flattens or rises: with 4 KB of text, the model *overfits*. Real pretraining sees trillions of tokens roughly once, so it rarely overfits this way; with small fine-tuning sets this curve is exactly what you monitor (early stopping, fewer epochs, more data).

## 8. After training: samples and next-token probabilities

In [ ]:
sample_after = generate(model, "Twinkle", n_new=160)
print("BEFORE training:\n" + sample_before)
print("\nAFTER training:\n" + sample_after)
print("\nAFTER training, prompt 'Jack':\n" + generate(model, "Jack", n_new=120, seed=1))

Compare the two. Before: uniformly random symbols. After a few hundred steps on 4 KB, a 108k-parameter model has learned the *shape* of the text: short lines, a capital letter after each newline, commas and full stops in the right places, spaces between word-sized chunks, many real words and fragments of the actual rhymes - but not yet coherent meaning. Everything beyond that (grammar, facts, reasoning) is what scale buys: more parameters, more data, more steps.

In [ ]:
prompt = "Twinkle, twinkle, little st"
with torch.no_grad():
    model.eval()
    logits, _ = model(torch.tensor([encode(prompt)]))
    probs = F.softmax(logits[0, -1], dim=-1).numpy()
top = np.argsort(-probs)[:10]
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar([repr(chars[i]) for i in top], probs[top], color="tab:green")
ax.set_ylabel("probability"); ax.set_title(f"Next-character distribution after {prompt!r}")
plt.show()
print("top-3:", [(chars[i], round(float(probs[i]), 3)) for i in top[:3]])
assert chars[top[0]] == "a", "the model should complete 'st' -> 'a' (star)"
assert np.isclose(probs.sum(), 1.0, atol=1e-5)

The model outputs a full probability distribution over the 57 characters at every step; generation is just "sample from it, append, repeat" (how to sample well is notebook 05). Everything a large LLM does at inference is this loop - with a 100k-token vocabulary, dozens of layers and billions of parameters.

## Try it yourself

**1.** Use `gpt_params` to compute GPT-2 medium (d=1024, 24 layers, context 1024, vocab 50257) and GPT-2 XL (d=1600, 48 layers). Which share of each is the token embedding?

In [ ]:
# Solution — try it yourself first, then run this
for name, d, L in [("GPT-2 medium", 1024, 24), ("GPT-2 XL", 1600, 48)]:
    p = gpt_params(50257, 1024, d, L)
    print(f"{name:13s} {p / 1e6:7.1f}M params | token embedding share {50257 * d / p:.0%}")

**2.** Where is the trained model confident? For one line of the training text, plot the probability the model assigns to each *actual* next character. Where is it unsure?

In [ ]:
# Solution — try it yourself first, then run this
line = "Humpty Dumpty sat on a wall,\nHumpty Dumpty had a great fall."
ids_line = torch.tensor([encode(line)])
with torch.no_grad():
    p_all = F.softmax(model(ids_line[:, :-1])[0][0], dim=-1)
p_true = p_all[torch.arange(len(line) - 1), ids_line[0, 1:]].numpy()
fig, ax = plt.subplots(figsize=(11, 2.8))
ax.bar(range(len(p_true)), p_true, color=plt.cm.RdYlGn(p_true))
ax.set_xticks(range(len(p_true)), [repr(c)[1:-1] for c in line[1:]], fontsize=7)
ax.set_ylabel("p(actual next char)"); ax.set_title("Confident inside words, less sure at the start of a word")
plt.show()
print(f"mean probability of the true next character: {p_true.mean():.2f}")

**3.** Count the extra parameters if you (a) untie the LM head and (b) replace both LayerNorms of every block with RMSNorm (no bias). Check (b) by building a block.

In [ ]:
# Solution — try it yourself first, then run this
untied_extra = gpt_params(V, CONTEXT, 64, 2, tied=False) - gpt_params(V, CONTEXT, 64, 2)
blk_ln = Block(64, 4)
blk_rms = Block(64, 4); blk_rms.ln1, blk_rms.ln2 = nn.RMSNorm(64), nn.RMSNorm(64)
print(f"(a) untying adds V*d = {untied_extra:,} parameters")
print(f"(b) RMSNorm block: {count(blk_rms):,} vs LayerNorm block {count(blk_ln):,} (saves 2 x d biases)")
assert count(blk_ln) - count(blk_rms) == 2 * 64

## Say it in an interview

- **A transformer in 30 seconds:** "Token ids become vectors via an embedding table; position is injected (learned table, or RoPE rotating q and k); then N identical blocks, each `x += attention(norm(x))` then `x += MLP(norm(x))`; a final norm; and the LM head turns each vector into logits over the vocabulary. Trained by next-token cross-entropy at every position in parallel, thanks to the causal mask."
- **Parameter formula:** 12·d² per block (4d² attention, 8d² MLP) plus embeddings V·d: GPT-2 small = 124M. The MLP is ~2/3 of the non-embedding weights; Llama's SwiGLU uses three matrices (gate, up, down) at ~3.5d.
- **Residual stream:** every sub-layer *adds* to a running vector; the identity path keeps gradients healthy in deep stacks and lets layers refine rather than replace.
- **Positions:** attention is order-blind; GPT-2 learns a table (hard length limit), RoPE rotates q/k so scores depend on relative offsets and can be stretched (NTK/YaRN) for longer contexts.
- **Weight tying:** head = embedding matrix transposed; saves V·d parameters (31% of GPT-2 small).
- **Training signal:** initial loss = ln(V) (uniform); a loss of L nats means perplexity e^L. Watch validation loss for overfitting on small data.
- **Traps:** post-norm vs pre-norm; forgetting that learned positions cap the context; counting tied weights twice; thinking the embedding of a word depends on context (it does not - the blocks add context).

## Next

- [05 · Decoding: temperature, top-k, top-p](05_decoding_temperature_top_k_top_p.ipynb) - how to turn the next-token distribution into text.
- [06 · KV cache and inference speed](06_kv_cache_and_inference_speed.ipynb) - make this model generate fast.
- [LoRA from scratch](../11_finetuning/03_lora_from_scratch.ipynb) - which of these matrices to adapt, and why.
- [Tokenizers, models and generate](../10_huggingface/02_tokenizers_models_and_generate.ipynb) - the same architecture in Hugging Face.
- Theory: [genai_flow course](../../genai_flow/index.html) (chapter 18 "One transformer block", 18b "Reading print(model)") · [interview bank](../../ai_interview_prep/index.html)